In [ ]:
#Reading data
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
plt.style.use('ggplot')


 
df = pd.read_csv(r'C:\Users\jamal\OneDrive\Personal\AI ML Purdue\2n project\AusApparalSales4thQrt2020.csv')
df_backup = df.copy()
df_test = df.copy()
df.head(50)

#### 1-	Data wrangling

In [ ]:
df.isna().sum()


In [ ]:
print("Rows with negative Unit values:")
display(df[df['Unit'] < 0])

print("\n Rows with negative Sales values:")
display(df[df['Sales'] < 0])

print("\nUnique Time values:", df['Time'].unique())
print("Unique State values:", df['State'].unique())
print("Unique Group values:", df['Group'].unique())

print("\nRows with zero or negative Sales:")
display(df[df['Sales'] <= 0])

In [ ]:
# Clean up whitespace in text columns
for col in ['Time', 'State', 'Group']:
    df[col] = df[col].str.strip()

# Check negative or zero values
neg_unit = df[df['Unit'] < 0]
neg_sales = df[df['Sales'] < 0]
zero_sales = df[df['Sales'] <= 0]

print("=== Data Quality Checks ===")
print(f"Negative Unit rows: {len(neg_unit)}")
print(f"Negative Sales rows: {len(neg_sales)}")
print(f"Zero or Negative Sales rows: {len(zero_sales)}\n")

print("Unique Time values:", df['Time'].unique())
print("Unique State values:", df['State'].unique())
print("Unique Group values:", df['Group'].unique())

if len(neg_unit) > 0:
    print("\nRows with negative Unit values:")
    display(neg_unit)

if len(neg_sales) > 0:
    print("\nRows with negative Sales values:")
    display(neg_sales)
print(df.dtypes)


#### D- d.	Share your insights regarding the application of the GroupBy() function for either data chunking or merging, and offer a recommendation based on your analysis.

In [ ]:
df_test['Date'] = pd.to_datetime(df_test['Date'], errors='coerce') #converting the date to a suitable type 
df_test.insert(1, 'DayName', df_test['Date'].dt.strftime('%a'))
df_test.head(10)
print(df_test.dtypes)



In [ ]:
avg_sales_per_day = df_test.groupby('DayName')['Sales'].mean()
print('Average Sales By Day')
print(avg_sales_per_day.round(2))
print('\n')
print('Average Sales By Day Per Group')
avg_sales_per_day_group = df_test.groupby(['DayName' ,'Group'])['Sales'].mean()
print(avg_sales_per_day_group.round(2))
print('\n')
print('Average Sales Per Group')
avg_sales_per_Group = df_test.groupby('Group')['Unit'].mean()
print(avg_sales_per_Group .round(2))
print('\n')
total_sales_per_state = df_test.groupby('State')['Sales'].sum()
print('Total Sales Per State')

print('\n', total_sales_per_state.round(2))


### 2-	Data Analysis

a.	Performing descriptive statistical analysis on the data in the Sales and Unit columns. Utilize techniques such as mean, median, mode, and standard deviation

In [ ]:
df.describe().round(2)

b.	Identifying the group with the highest sales and the group with the lowest sales based on the data provided.

In [ ]:
sales_by_group = df.groupby(['Group'])[['Sales']].sum().sort_values('Sales')
highest_lowest = pd.concat([
    sales_by_group.nlargest(1, 'Sales'),
    sales_by_group.nsmallest(1, 'Sales') ])

print(highest_lowest)



c.	Vistulizing the group with the highest sales and the group with the lowest sales based on the data provided.

In [ ]:
highest_lowest.plot(kind='barh', figsize=(10,2), legend=False, color=['blue'] , width=0.5)
plt.ticklabel_format(style='plain', axis='x')
plt.grid(axis='x', linestyle='--', alpha=0.7) 
plt.title('Highest and Lowest Sales by Group')
plt.xlabel('Total Sales')
plt.ylabel('Group')
plt.xticks(ticks=range(0, 90000000, 10000000), rotation=0)
for i, v in enumerate(highest_lowest['Sales']):
    plt.text(v + 5000, i, f"{v:,}", va='center' , ha='right', color='white')
plt.show()


4. d.	Generating weekly, monthly, and quarterly reports to document and present the results of the analysis conducted.

In [ ]:
df_dates = df.copy()
print (df_dates.dtypes) #testing the type of Date column
df_dates['Date'] = pd.to_datetime(df_dates['Date']) #converting Date ti date time type
print (df_dates.dtypes)
df_dates.insert(1, 'Year', df_dates['Date'].dt.year)
df_dates.insert(2, 'Month', df_dates['Date'].dt.month)
df_dates.insert(3, 'Day', df_dates['Date'].dt.day)
df_dates.insert(4, 'Week', df_dates['Date'].dt.isocalendar().week)
df_dates.head(10)



In [ ]:
Weekly_Sale_By_group = df_dates.groupby(['Week' ,'Group'])['Sales'].sum().reset_index().rename(columns={'Sales': 'Total_Sales'})  
print('Total Weekly Sales\n' , Weekly_Sale_By_group)
Monthly_Sale_By_group = df_dates.groupby(['Month' ,'Group'])['Sales'].sum().reset_index().rename(columns={'Sales': 'Total_Sales'})  
print('\nTotal Monthly Sales\n' , Monthly_Sale_By_group)
Quarterly_Sale_By_group = df_dates.groupby(['Group'])['Sales'].sum() #since the data are for oct thru Dec, total will represent quarter otherwise we could use something like resample('QE').sum
print('\nTotal Quarerly Sales' , Quarterly_Sale_By_group)

In [ ]:
pivot_df = Weekly_Sale_By_group.pivot(index='Week', columns='Group', values='Total_Sales')
pivot_df.plot(marker='o', figsize=(12,6))
plt.ticklabel_format(style='plain', axis='y')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.title('Weekly Sales by Group')
plt.xlabel('Week')
plt.ylabel('Total Sales')
plt.xticks(ticks=range(df_dates['Week'].min(), df_dates['Week'].max()+1, 1))

plt.show()


plt.figure(figsize=(14,7))
sns.barplot(x='Group', y='Sales', hue='State', data=df_dates)
plt.title('Group-wise Sales across States')
plt.xticks(rotation=45)
plt.show()

plt.figure(figsize=(14,7))
sns.barplot(x='State', y='Sales', hue='State', data=df_dates)
plt.title(' Total Sales Per States')
plt.xticks(rotation=45)
plt.show()

#### Observations:
    - sales over time follow the same trends for all groups 
    - In terms of demographic groups , Men spend the most, while Seniors spend the least.
    - lower sales in the last week of the year as most people tend to buy before that week 
    In terms of States sales volumes: VIC is the highest , WA is the lowest

In [ ]:
pivot_df = Monthly_Sale_By_group.pivot(index='Month', columns='Group', values='Total_Sales')
pivot_df.plot(marker='o', figsize=(12,6))
plt.ticklabel_format(style='plain', axis='y')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.title('Monthly Sales by Group')
plt.xlabel('Month')
plt.ylabel('Total Sales')
plt.xticks(ticks=range(df_dates['Month'].min(), df_dates['Month'].max()+1))

plt.show()

In [ ]:
Quarterly_Sale_By_group.plot(kind='barh', figsize=(10,2), legend=False, color=['blue'] , width=0.5 )
Quarterly_Sale_By_group = Quarterly_Sale_By_group.sort_values(ascending=True)
plt.ticklabel_format(style='plain', axis='x')
plt.grid(axis='x', linestyle='--', alpha=0.7) 
plt.title('Quarterly  Sales by Group')
plt.xlabel('Total Sales')
plt.ylabel('Group')
plt.xticks(ticks=range(0, 90000000, 10000000), rotation=0)
for i, v in enumerate(Quarterly_Sale_By_group):
    plt.text(v + 50000, i, f"{v:,}", va='center' , ha='right', color='white')
plt.show()

#### Observations:
    - sales over time follow the same trends for all groups 
    - Although the weekly sales decline in the last week in the year but the total monthly and quarterly still higher 
    - People makes their purchases prior the end of the year, at least one or two weeks

### 3-	Data Visualization

State-wise sales analysis for different demographic groups

In [ ]:
states = df_dates['State'].unique()
groups = df_dates['Group'].unique() 


fig, axes = plt.subplots(nrows=7, ncols=4, figsize=(20, 30))
axes = axes.flatten()  # flatten to 1D array for easy indexing

for i, state in enumerate(states):
    for j, group in enumerate(groups):
        idx = i*4 + j  # calculate 1D index for flattened axes
        ax = axes[idx]
                  
        # Plot
        sns.barplot(data=Weekly_Sale_By_group, x='Week', y='Total_Sales', ax=ax, color='skyblue', errorbar=None)
        ax.set_title(f"{state} - {group}")
        ax.set_xlabel('Week')
        ax.set_ylabel('Total Sales')
        ax.ticklabel_format(style='plain', axis='y')  # avoids scientific notation

plt.tight_layout()
plt.show()

In [ ]:
states = df_dates['State'].unique()
groups = df_dates['Group'].unique() 


fig, axes = plt.subplots(nrows=7, ncols=4, figsize=(20, 30))
axes = axes.flatten()  # flatten to 1D array for easy indexing

for i, state in enumerate(states):
    for j, group in enumerate(groups):
        idx = i*4 + j  # calculate 1D index for flattened axes
        ax = axes[idx]
                  
        # Plot
        sns.barplot(data=Monthly_Sale_By_group, x='Month', y='Total_Sales', ax=ax, color='skyblue', errorbar=None)
        ax.set_title(f"{state} - {group}")
        ax.set_xlabel('Month')
        ax.set_ylabel('Total Sales')
        ax.ticklabel_format(style='plain', axis='y')  # avoids scientific notation

plt.tight_layout()
plt.show()


Group-wise sales analysis (Kids, Women, Men, and Seniors) across various states - Weekly Sales 

In [ ]:
fig, axes = plt.subplots(nrows=4, ncols=7, figsize=(40, 20))
axes = axes.flatten()  # flatten to 1D array for easy indexing

for i, group in enumerate(groups):
    for j, state in enumerate(states):
        idx = i*7 + j  # calculate 1D index for flattened axes
        ax = axes[idx]
                  
        # Plot
        sns.barplot(data=Weekly_Sale_By_group, x='Week', y='Total_Sales', ax=ax, color='skyblue', errorbar=None)
        ax.set_title(f"{state} - {group}")
        ax.set_xlabel('Week')
        ax.set_ylabel('Total Sales')
        ax.ticklabel_format(style='plain', axis='y')  # avoids scientific notation

plt.tight_layout()
plt.show()

Group-wise sales analysis (Kids, Women, Men, and Seniors) across various states.

In [ ]:
df_group_wise = df_backup.copy()
ct_group_wise = pd.crosstab(index=df_group_wise['Group'], columns=df_group_wise['State'] , values=df_group_wise['Sales'], aggfunc='sum' )
ax = ct_group_wise.plot(kind='bar', stacked=True, figsize=(10,6))

for container in ax.containers:
    ax.bar_label(container, fmt='{:,.0f}', label_type='center', color='white', fontsize=9)


plt.ticklabel_format(style='plain', axis='y')
plt.title('Total Sales by Group and State', fontsize=14)
plt.xlabel('Group', fontsize=12)
plt.ylabel('Total Sales', fontsize=12)
plt.legend(title='State', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.yticks(np.arange(0, 90000000, 10000000))
plt.tight_layout()
plt.show()


Time-of-the-day analysis

In [ ]:

df_day_time = df_backup.copy() #creating a fresh copy of df

# Convert Date to datetime for consistency
df_day_time['Date'] = pd.to_datetime(df['Date'], format='%d-%b-%Y', errors='coerce')



# calculating Total Sales by Time of Day
day_time_sales = df.groupby('Time', as_index=False)['Sales'].sum().sort_values('Sales', ascending=False)
print("\nTotal Sales by Time of Day:\n", day_time_sales)

# Visualization – Total Sales by Time

plt.figure(figsize=(8,5))
sns.barplot(data=day_time_sales, x='Time', y='Sales', hue='Time', palette='viridis', legend=False)
plt.title('Total Sales by Time of Day', fontsize=14)
plt.xlabel('Time of Day')
plt.ylabel('Total Sales ($)')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.ticklabel_format(style='plain', axis='y')
plt.show()

# Identify Peak & Off-Peak
peak = day_time_sales.loc[day_time_sales['Sales'].idxmax()]
offpeak = day_time_sales.loc[day_time_sales['Sales'].idxmin()]

print(f"\n Peak Sales Period: {peak['Time']} with ${peak['Sales']:,}")
print(f" Off-Peak Sales Period: {offpeak['Time']} with ${offpeak['Sales']:,}")

# Breakdown by Group
group_time = df.groupby(['Group', 'Time'], as_index=False)['Sales'].sum()

plt.figure(figsize=(10,6))
sns.lineplot(data=group_time, x='Time', y='Sales', hue='Group', marker='o')
plt.title('Sales Trend by Group and Time of Day')
plt.xlabel('Time of Day')
plt.ylabel('Total Sales ($)')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()


# Breakdown by State

state_time = df.groupby(['State', 'Time'], as_index=False)['Sales'].sum()

plt.figure(figsize=(10,6))
sns.lineplot(data=state_time, x='Time', y='Sales', hue='State', marker='o')
plt.title('Sales Trend by State and Time of Day')
plt.xlabel('Time of Day')
plt.ylabel('Total Sales ($)')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()



#### Observations:
- Peak sales occur during the Morning period.
- Off-peak sales occur during the Evening period.
- Marketing and staffing can be optimized around these times for maximum impact.
- Use this data to guide hyper-personalized offers and Next Best Offer timing.